# step2_ncf_nmf

**Owner:** P3  
**Purpose:** NCF / NMF (the course baselines) and cold-versus-warm gap diagnosis.  
**Inputs:** `data/processed/` from step 1  
**Outputs:** `outputs/results_step2_ncf_nmf.csv`

> Runtime: local VS Code/Jupyter. Install `requirements.txt`, select that Python environment as the notebook kernel, and run all cells top to bottom. Training on MIND-large may be slow without a GPU.

In [ ]:
# --- Setup: run this first, every session ---
from google.colab import drive
drive.mount('/content/drive')

!rm -rf /content/BT4222-Project-Group-4--MIND && git clone -q https://github.com/Fdiannz/BT4222-Project-Group-4--MIND.git /content/BT4222-Project-Group-4--MIND
import sys; sys.path.insert(0, '/content/BT4222-Project-Group-4--MIND')

import numpy as np, pandas as pd
import config
from utils.data_utils import load_news, load_behaviors, explode_impressions
from utils.metrics import evaluate, evaluate_by_segment, topk, topk_diversity
print("Data folder:", config.DATA_DIR)

In [ ]:
import copy, time
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

SMOKE = False   # True = tiny run (1 epoch, small subsets) to check the whole pipeline in a few minutes
SEED = config.SEED
np.random.seed(SEED); torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
if DEVICE == "cpu" and not SMOKE:
    print("WARNING: no GPU found. Switch the runtime to a GPU or set SMOKE = True.")

# CHANGED: column names this notebook expects. If step 1 uses different names, add them to RENAME, not in the code below.
USER_COL, ITEM_COL, COLD_COL = "user_id", "news_id", "is_cold"
RENAME = {}   # {"actual_column_in_parquet": "expected_name"}, e.g. {"clicked": "label", "cold_flag": "is_cold"}

# CHANGED: step 1 output files in data/processed/
FILES = {
    "fit":       "fit_candidates_sampled.parquet",   # 9-13 Nov, 1:4 negative sampled
    "tune":      "tune_candidates.parquet",          # 14 Nov, complete slates
    "test":      "eval_candidates.parquet",          # 15 Nov, complete slates (official dev)
    "news":      "news_metadata.parquet",
}

PROC, OUT = Path(config.PROCESSED_DIR), Path(config.OUTPUT_DIR)
SCORES = Path(getattr(config, "SCORES_DIR", OUT / "scores"))      # per-candidate score files, one per model
SCORES.mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)

REQUIRED_STEP1_FILES = [FILES[key] for key in ("fit", "tune", "test", "news")]
missing_files = [PROC / name for name in REQUIRED_STEP1_FILES if not (PROC / name).is_file()]
if missing_files:
    missing = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(
        "Cleaned Step 1 parquet files are missing. Run all cells in "
        "step1_data_preprocess.ipynb first; it writes the candidate splits and "
        f"news metadata to {PROC}. If your data is elsewhere, set MIND_PROJECT_DIR "
        f"before starting the kernel.\nMissing files:\n{missing}"
    )

# P3's own modelling settings (shared settings stay in config.py)
BATCH = 8192
MAX_EPOCHS = 1 if SMOKE else 6
PATIENCE = 1                                   # stop after this many epochs without tune improvement
TUNE_MAX_IMPRESSIONS = 5_000 if SMOKE else 100_000    # 🔧 I6: complete impressions, never partial slates
TRAIN_EVAL_IMPRESSIONS = 5_000 if SMOKE else 200_000
SCORE_BATCH = 262_144

## 1. NCF / NMF with user and article ID embeddings

Loads the processed splits from step 1, checks that they match what the interim report promises, and builds the ID vocabulary.

**Why.**
- *Leak-free IDs (🔧 I7).* The user and article vocabularies are built from the **fit split only**. Anything that appears first in tune or test (a new article, a new user) maps to index 0, the "unknown" slot. This is exactly what would happen in production, and it is the mechanism behind the cold-start failure we want to diagnose.
- *Sanity checks (🔧 I7).* Fit must be negative-sampled (≤ 4 negatives per click), tune and test must be **full** slates (~4% clicks), and time must be ordered fit < tune < test. If a check fails, the problem is in the processed files, not in the model.

In [ ]:
# Update RENAME to map 'cold_article' in the parquet files to the expected COLD_COL name ('is_cold')
RENAME.update({"cold_article": COLD_COL})

def read_split(key, extra=()):
    path = PROC / FILES[key]
    present = pq.read_schema(path).names
    std = {RENAME.get(c, c): c for c in present}              # expected name -> actual name in the file
    need = ["impression_id", USER_COL, ITEM_COL, "label", *extra]
    missing = [c for c in need if c not in std]
    if missing:
        raise ValueError(f"{path} is missing required columns {missing}. "
                         f"Columns present: {present}. Fix RENAME or check the Step 1 output schema.")
    keep = need + [c for c in ["time"] if c in std]
    return pd.read_parquet(path, columns=[std[c] for c in keep]).rename(columns=RENAME)

fit  = read_split("fit")
tune = read_split("tune", extra=[COLD_COL])
test = read_split("test", extra=[COLD_COL])
news = pd.read_parquet(PROC / FILES["news"]).rename(columns=RENAME)
assert ITEM_COL in news.columns, (f"{FILES['news']} has no `{ITEM_COL}` column (columns: {list(news.columns)}). "
                                  "If it only has an integer index, it must map to the same news_id strings used in the candidates.")
for d in (tune, test):
    d[COLD_COL] = d[COLD_COL].astype(np.int8)
for d in (fit, tune, test):
    d["label"] = d["label"].astype(np.int8)

for name, d in [("fit", fit), ("tune", tune), ("test", test)]:
    print(f"{name:5s} rows={len(d):>12,}  impressions={d['impression_id'].nunique():>9,}  "
          f"users={d[USER_COL].nunique():>8,}  click rate={d['label'].mean():.4f}")

# --- checks against the interim report ---
neg_per_pos = (fit["label"] == 0).sum() / (fit["label"] == 1).sum()
print(f"\nfit negatives per click: {neg_per_pos:.2f}   (config.NEG_PER_POS = {config.NEG_PER_POS})")
assert neg_per_pos <= config.NEG_PER_POS + 0.01, "fit has more than NEG_PER_POS negatives per click"
for name, d in [("tune", tune), ("test", test)]:
    assert d["label"].mean() < 0.10, (f"{name} click rate {d['label'].mean():.3f} is far above the ~4% of full "
                                      "slates: it looks negative-sampled. Evaluation must use complete slates.")
if all("time" in d for d in (fit, tune, test)):
    assert fit["time"].max() < tune["time"].min() and tune["time"].max() < test["time"].min(), "splits overlap in time"
    print("time order OK: fit < tune < test")
else:
    print("no `time` column in the processed files: time order not checked here")
print("cold share  tune:", round(tune[COLD_COL].mean(), 4), "  test:", round(test[COLD_COL].mean(), 4))

In [ ]:
# Vocabulary from the FIT split only; index 0 is reserved for "unknown / never seen in fit"
user_keys = pd.Index(fit[USER_COL].unique())
item_keys = pd.Index(fit[ITEM_COL].unique())
n_users, n_items = len(user_keys), len(item_keys)

def add_ids(d):
    d["u"] = (user_keys.get_indexer(d[USER_COL]) + 1).astype(np.int32)
    d["i"] = (item_keys.get_indexer(d[ITEM_COL]) + 1).astype(np.int32)
    d["oov_user"] = d["u"] == 0
    d["oov_item"] = d["i"] == 0
    return d

fit, tune, test = add_ids(fit), add_ids(tune), add_ids(test)
assert not fit["oov_user"].any() and not fit["oov_item"].any()
print(f"vocabulary from fit: {n_users:,} users, {n_items:,} articles")
for name, d in [("tune", tune), ("test", test)]:
    print(f"{name}: {d['oov_item'].mean():.1%} of displays and {d['oov_user'].mean():.1%} of rows have an ID not seen in fit")

In [ ]:
# Cross-check against the interim report (Section 3.4): dev articles never shown in the training period.
# Approximate: `fit` is negative-sampled, so an article shown in 9-13 Nov but never sampled counts as unseen here.
seen_period = pd.Index(pd.concat([fit[ITEM_COL], tune[ITEM_COL]]).unique())
row_unseen = ~test[ITEM_COL].isin(seen_period)
art_unseen = ~pd.Index(test[ITEM_COL].unique()).isin(seen_period)
print(pd.DataFrame({
    "this data": [art_unseen.mean(), row_unseen.mean(), row_unseen[test["label"] == 1].mean()],
    "interim report": [0.323, 0.054, 0.088]},
    index=["dev articles never shown in 9-14 Nov", "share of dev displays", "share of dev clicks"]).apply(lambda col: col.map("{:.1%}".format)))

## 2. Training on the fit split; tuning on 14 Nov

Defines the three ID-only models, trains them on the negative-sampled fit split with binary cross-entropy, and uses 14 Nov (complete slates) for early stopping and hyperparameter selection. The 15 Nov test split is not touched here.

**Why these models (Topic 6).**
- **GMF** = matrix factorisation: score = a learned function of the element-wise product of the user and article vectors.
- **NCF-MLP** = concatenate the two vectors and pass them through an MLP (ReLU, dropout).
- **NeuMF** = both towers with separate embeddings, concatenated before the output layer.

**Design choices to defend in the report.**
- *Unknown IDs.* Index 0 is a **zero vector** in the baselines and is never updated, so an unseen article is scored with no ID information at all. This is the honest "ID-only" behaviour.
- *🔧 I5 ID masking (Topic 6).* In the ablation, each ID is replaced by index 0 with probability `p` during training, so the unknown slot learns a generic "average user / average article" vector. If cold results improve a lot, the cold failure is mostly an untrained-unknown-vector problem. If they barely move, ID-only models lack the information, which is the argument for P2/P4/P5's content features.
- *🔧 I9 Embedding L2 on the batch.* Adam's `weight_decay` adds the penalty to every embedding row at every step, including rows not in the batch, which shrinks rarely seen articles the most. We penalise only the embeddings used in the batch instead (Topic 3, L2 regularisation).
- *🔧 I6 Early stopping on nDCG@10* over complete 14 Nov impressions. Group 10 found AUC peaking early while ranking metrics kept improving, so we select on a ranking metric and also log AUC.
- Batch norm is deliberately left out: Topic 9A flags dropout-before-batch-norm as a variance-shift issue, and this MLP is shallow.


In [ ]:
class NeuMF(nn.Module):
    """GMF + MLP towers (He et al. 2017). use_gmf/use_mlp switch it into plain GMF or plain NCF-MLP.
    Index 0 of every embedding table is the 'unknown ID' slot."""
    def __init__(self, n_users, n_items, gmf_dim=32, mlp_dim=32, layers=(64, 32, 16),
                 dropout=0.2, use_gmf=True, use_mlp=True, zero_oov=True):
        super().__init__()
        self.use_gmf, self.use_mlp = use_gmf, use_mlp
        embs = []
        if use_gmf:
            self.gmf_u, self.gmf_i = nn.Embedding(n_users + 1, gmf_dim), nn.Embedding(n_items + 1, gmf_dim)
            embs += [self.gmf_u, self.gmf_i]
        if use_mlp:
            self.mlp_u, self.mlp_i = nn.Embedding(n_users + 1, mlp_dim), nn.Embedding(n_items + 1, mlp_dim)
            embs += [self.mlp_u, self.mlp_i]
            dims = [2 * mlp_dim, *layers]
            self.mlp = nn.Sequential(*[m for a, b in zip(dims[:-1], dims[1:])
                                       for m in (nn.Linear(a, b), nn.ReLU(), nn.Dropout(dropout))])
        self.out = nn.Linear((gmf_dim if use_gmf else 0) + (layers[-1] if use_mlp else 0), 1)
        for e in embs:
            nn.init.normal_(e.weight, std=0.01)
            if zero_oov:                       # baseline: unknown ID = zero vector, never updated
                with torch.no_grad():
                    e.weight[0].zero_()

    def forward(self, u, i, return_reg=False):
        feats, reg = [], 0.0
        if self.use_gmf:
            pu, qi = self.gmf_u(u), self.gmf_i(i)
            feats.append(pu * qi)
            reg = reg + pu.pow(2).sum() + qi.pow(2).sum()
        if self.use_mlp:
            pu, qi = self.mlp_u(u), self.mlp_i(i)
            feats.append(self.mlp(torch.cat([pu, qi], dim=1)))
            reg = reg + pu.pow(2).sum() + qi.pow(2).sum()
        logit = self.out(torch.cat(feats, dim=1)).squeeze(1)
        return (logit, reg / u.shape[0]) if return_reg else logit


def make_cfg(dim=32, l2=1e-4, id_mask=0.0, gmf=True, mlp=True, dropout=0.2, lr=1e-3):
    return dict(arch=dict(gmf_dim=dim, mlp_dim=dim, use_gmf=gmf, use_mlp=mlp, dropout=dropout),
                l2=l2, id_mask=id_mask, lr=lr)

In [ ]:
@torch.no_grad()
def score_ids(model, u_idx, i_idx):
    """Probability that each (user, article) pair is clicked. Batched so 14M rows fit in memory."""
    model.eval()
    out = []
    for s in range(0, len(u_idx), SCORE_BATCH):
        u = torch.tensor(u_idx[s:s + SCORE_BATCH], device=DEVICE, dtype=torch.long)
        i = torch.tensor(i_idx[s:s + SCORE_BATCH], device=DEVICE, dtype=torch.long)
        out.append(torch.sigmoid(model(u, i)).float().cpu())
    return torch.cat(out).numpy()


def base_eval(d):
    """Candidate table in the layout utils.metrics expects (impression_id, news_id, label[, is_cold])."""
    cols = ["impression_id", ITEM_COL, "label"] + ([COLD_COL] if COLD_COL in d else [])
    return d[cols].rename(columns={ITEM_COL: "news_id"}).copy()


def make_eval_frame(d, model):
    ev = base_eval(d)
    ev["score"] = score_ids(model, d["u"].to_numpy(), d["i"].to_numpy())
    return ev

def attach_scores(candidates, scores):
    """Validate candidate-score alignment before handing scores to another step."""
    key_cols = ["impression_id", "news_id"]
    missing_candidates = [col for col in key_cols if col not in candidates]
    missing_scores = [col for col in [*key_cols, "score"] if col not in scores]
    if missing_candidates or missing_scores:
        raise ValueError(f"Missing score contract columns: candidates={missing_candidates}, scores={missing_scores}")
    if candidates[key_cols].isna().any().any():
        raise ValueError("Candidate keys contain missing values.")
    if candidates.duplicated(key_cols).any():
        raise ValueError("Candidate table contains duplicate impression/article pairs.")
    aligned = candidates[key_cols].reset_index(drop=True).equals(scores[key_cols].reset_index(drop=True))
    if not aligned:
        raise ValueError("Scores are missing, extra, or out of order relative to the candidate table.")
    values = pd.to_numeric(scores["score"], errors="raise").to_numpy(dtype=np.float64)
    if not np.isfinite(values).all():
        raise ValueError("Scores contain NaN or infinite values.")
    return scores[key_cols].assign(score=values)

# Fallback: dynamically add mapped columns 'u' and 'i' if they are missing in the current session
if "u" not in fit.columns or "i" not in fit.columns:
    user_keys = pd.Index(fit[USER_COL].unique())
    item_keys = pd.Index(fit[ITEM_COL].unique())
    def add_ids(d):
        d["u"] = (user_keys.get_indexer(d[USER_COL]) + 1).astype(np.int32)
        d["i"] = (item_keys.get_indexer(d[ITEM_COL]) + 1).astype(np.int32)
        d["oov_user"] = d["u"] == 0
        d["oov_item"] = d["i"] == 0
        return d
    fit = add_ids(fit)
    tune = add_ids(tune)
    test = add_ids(test)

fit_u = torch.tensor(fit["u"].to_numpy(copy=True), device=DEVICE, dtype=torch.long)
fit_i = torch.tensor(fit["i"].to_numpy(copy=True), device=DEVICE, dtype=torch.long)
fit_y = torch.tensor(fit["label"].to_numpy(dtype=np.float32, copy=True), device=DEVICE)

# 🔧 I6: tune on a random subset of COMPLETE impressions of 14 Nov (never a subset of candidates)
tune_ids = tune["impression_id"].unique()
sub_ids = rng.choice(tune_ids, size=min(TUNE_MAX_IMPRESSIONS, len(tune_ids)), replace=False)
tune_sub = tune[tune["impression_id"].isin(sub_ids)]
print(f"fit rows on device: {len(fit_y):,}   tune subset: {len(sub_ids):,} of {len(tune_ids):,} impressions")

In [ ]:
def train_model(name, cfg, max_epochs=MAX_EPOCHS, verbose=True):
    torch.manual_seed(SEED)
    arch = {**cfg["arch"], "zero_oov": cfg["id_mask"] == 0}
    model = NeuMF(n_users, n_items, **arch).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
    n, best_state, best, bad, hist = len(fit_y), None, -1.0, 0, []
    for ep in range(1, max_epochs + 1):
        model.train()
        t0, perm, tot = time.time(), torch.randperm(n, device=DEVICE), torch.zeros((), device=DEVICE)
        for s in range(0, n, BATCH):
            idx = perm[s:s + BATCH]
            u, i, y = fit_u[idx].long(), fit_i[idx].long(), fit_y[idx]
            if cfg["id_mask"] > 0:             # 🔧 I5: ID masking, the unknown slot learns a generic vector
                u = u.masked_fill(torch.rand(len(u), device=DEVICE) < cfg["id_mask"], 0)
                i = i.masked_fill(torch.rand(len(i), device=DEVICE) < cfg["id_mask"], 0)
            logit, reg = model(u, i, return_reg=True)
            loss = F.binary_cross_entropy_with_logits(logit, y) + cfg["l2"] * reg
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.detach() * len(idx)
        m = evaluate(make_eval_frame(tune_sub, model), "score")
        hist.append({"epoch": ep, "train_loss": (tot / n).item(), "tune_auc": m["auc"],
                     "tune_mrr": m["mrr"], "tune_ndcg@10": m["ndcg@10"]})
        if verbose:
            print(f"[{name}] epoch {ep}: loss {hist[-1]['train_loss']:.4f}  tune AUC {m['auc']:.4f}  "
                  f"nDCG@10 {m['ndcg@10']:.4f}  ({time.time() - t0:.0f}s)")
        if m["ndcg@10"] > best:
            best, bad, best_state = m["ndcg@10"], 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad > PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist), best

**Hyperparameter search (small grid, NeuMF only).** Embedding size × batch-L2 strength, selected on 14 Nov nDCG@10 over complete impressions. The learning rate (1e-3, Adam) follows the NCF paper. The selected size and L2 are reused for GMF and NCF-MLP so the three variants stay comparable.

In [ ]:
GRID = [
    make_cfg(dim=d, l2=l2)
    for d in (32, 64)
    for l2 in (1e-4, 1e-3)
]

if SMOKE:
    GRID = GRID[:1]

models, histories, grid_rows = {}, {}, []

best_score = -1.0
best_cfg, best_model, best_hist = None, None, None

for i, cfg in enumerate(GRID, 1):
    dim = cfg["arch"]["gmf_dim"]
    l2 = cfg["l2"]

    tag = f"NeuMF dim={dim} l2={l2:g}"

    print(f"\n[{i}/{len(GRID)}] Training {tag}")

    model, hist, score_ = train_model(tag, cfg)

    best_epoch = int(hist["tune_ndcg@10"].idxmax()) + 1

    grid_rows.append({
        "config": tag,
        "tune_ndcg@10": score_,
        "best_epoch": best_epoch
    })

    if score_ > best_score:
        # Keep the best model
        if best_model is not None:
            del best_model

        best_score = score_
        best_cfg = cfg
        best_model = model
        best_hist = hist
    else:
        del model

    print(f"Score: {score_:.4f} | Best epoch: {best_epoch}")

grid = pd.DataFrame(grid_rows).sort_values(
    "tune_ndcg@10", ascending=False
)

display(grid)

BEST_DIM = best_cfg["arch"]["gmf_dim"]
BEST_L2 = best_cfg["l2"]

models["NeuMF"] = best_model
histories["NeuMF"] = best_hist

print("Selected:", BEST_DIM, BEST_L2)

**Remaining variants.** GMF and NCF-MLP use the selected size and L2. The two ID-masking variants are NeuMF with the same settings plus masking (`p` = 0.1 and 0.3). They are the ablation for the gap diagnosis (🔧 I5).

In [ ]:
VARIANTS = {
    "GMF (MF)": make_cfg(BEST_DIM, BEST_L2, gmf=True, mlp=False),
    "NCF-MLP": make_cfg(BEST_DIM, BEST_L2, gmf=False, mlp=True),
    "NeuMF + ID masking (p=0.1)": make_cfg(BEST_DIM, BEST_L2, id_mask=0.1),
    "NeuMF + ID masking (p=0.3)": make_cfg(BEST_DIM, BEST_L2, id_mask=0.3),
}
if SMOKE:
    VARIANTS = dict(list(VARIANTS.items())[:3])
for name, cfg in VARIANTS.items():
    models[name], histories[name], _ = train_model(name, cfg)
print("trained:", list(models))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for name, h in histories.items():
    ax[0].plot(h["epoch"], h["train_loss"], marker="o", label=name)
    ax[1].plot(h["epoch"], h["tune_ndcg@10"], marker="o", label=name)
ax[0].set(title="Training loss (fit, 1:4 pools)", xlabel="epoch", ylabel="BCE + L2")
ax[1].set(title="Tune nDCG@10 (14 Nov, complete slates)", xlabel="epoch", ylabel="nDCG@10")
ax[1].legend(fontsize=7)
plt.tight_layout(); plt.savefig(OUT / "figures" / "step2_ncf_nmf_training_curves.png", dpi=150); plt.show()

## 3. 📊 Results: train vs test, overall and cold vs warm

Scores every model on three sets and reports the shared metrics (per-impression AUC, MRR, nDCG@5/10):


| Set | What it is | How to read it |
|---|---|---|
| **train** | a random sample of fit impressions | still 1:4 negative-sampled pools (≈5 candidates per click), so these numbers are **not comparable** with full-slate numbers. Use them only to see whether a model fits the training data much better than the held-out days. |
| **tune** | 14 Nov, complete slates (🔧 I1) | the model-selection set |
| **test** | 15 Nov (official dev), complete slates | the final numbers; overall and split by whether the **clicked** article is cold, warm, or mixed |

**Why.** The interim report reports relevance separately for cold and warm clicks (§1.2). Every candidate is scored, including articles and users unseen in fit (🔧 I1, I4), and the shared metrics break score ties randomly (🔧 I2), so a model that outputs many equal scores is neither helped nor hurt.

A **random-score reference** is added so the cold numbers can be read against a floor. P1's popularity notebook holds the official random and popularity baselines.

In [ ]:
train_ids = rng.choice(fit["impression_id"].unique(), size=min(TRAIN_EVAL_IMPRESSIONS, fit["impression_id"].nunique()), replace=False)
fit_eval = fit[fit["impression_id"].isin(train_ids)]

rows, test_scores = [], {}
def add_rows(name, split, df):
    if isinstance(df, dict):
        df = pd.DataFrame({"all": df}).T
    for seg, r in df.iterrows():
        rows.append({"model": name, "split": split, "segment": seg, **r.to_dict()})

for name, model in models.items():
    add_rows(name, "train (fit sample, 1:4 pools)", evaluate(make_eval_frame(fit_eval, model), "score"))
    add_rows(name, "tune (14 Nov, full slates)", evaluate(make_eval_frame(tune, model), "score"))
    ev = make_eval_frame(test, model)
    test_scores[name] = ev["score"].to_numpy()
    add_rows(name, "test (15 Nov, full slates)", evaluate_by_segment(ev, "score", COLD_COL))
    print("done:", name)

# random reference, same metrics, same test slates
ev = base_eval(test)
ev["score"] = np.random.default_rng(SEED).random(len(ev))
test_scores["Random (reference)"] = ev["score"].to_numpy()
add_rows("Random (reference)", "test (15 Nov, full slates)", evaluate_by_segment(ev, "score", COLD_COL))

results = pd.DataFrame(rows)
results[["n_impressions", "n_skipped"]] = results[["n_impressions", "n_skipped"]].astype(int)
results.to_csv(OUT / "results_step2_ncf_nmf.csv", index=False)
print("saved", OUT / "results_step2_ncf_nmf.csv")

In [ ]:
t = results[results["split"].str.startswith("test")]
for metric in ["ndcg@10", "mrr", "auc"]:
    print(f"\n=== TEST {metric} by segment of the clicked article ===")
    display(t.pivot(index="model", columns="segment", values=metric).reindex(columns=["all", "cold", "warm", "mixed"]).round(4))
print("\n=== impressions per segment (test) ===")
display(t[t["model"] == "NeuMF"].set_index("segment")[["n_impressions", "n_skipped"]])

print("\n=== train vs tune vs test (overall) ===")
display(results[results["segment"] == "all"].pivot(index="model", columns="split", values="ndcg@10").round(4))

In [ ]:
pv = t.pivot(index="model", columns="segment", values="ndcg@10").reindex(columns=["all", "cold", "warm"])
ax = pv.plot(kind="bar", figsize=(10, 3.8), rot=20)
ax.set(title="Test nDCG@10 by segment of the clicked article (full slates)", ylabel="nDCG@10")
plt.tight_layout(); plt.savefig(OUT / "figures" / "step2_ncf_nmf_cold_vs_warm.png", dpi=150); plt.show()

**Hand-off to P6 (🔧 I8).** The test scores of the tuned NeuMF and of the best ID-masked variant are saved to `config.SCORES_DIR` in the shared contract: one row per displayed candidate with `impression_id, news_id, score`. `attach_scores` raises if any candidate is missing, duplicated or NaN, so a silent gap cannot make a model look better than it is. `score` is the sigmoid probability. It was trained with negative sampling, so use it for ranking, not as a calibrated probability. P6 should normalise it within each impression before mixing it with similarities in MMR.

In [ ]:
tune_full = results[(results["split"].str.startswith("tune")) & results["model"].str.startswith("NeuMF + ID")]
best_masked = tune_full.sort_values("ndcg@10", ascending=False)["model"].iloc[0] if len(tune_full) else None
labels_test = base_eval(test)[["impression_id", "news_id", "label"]]
for name in ["NeuMF", best_masked]:
    if name is None:
        continue
    sc = labels_test[["impression_id", "news_id"]].assign(score=test_scores[name])
    sc = attach_scores(labels_test, sc)                     # raises on any gap, duplicate or NaN
    slug = name.lower().replace(" ", "_").replace("(", "").replace(")", "").replace("=", "").replace("+", "plus")
    sc.to_parquet(SCORES / f"ncf_nmf_{slug}.parquet", index=False)
    print("saved score file for", name)

## 4. Gap diagnosis: why these models do badly on cold articles

Each sub-step answers one question for report aspect 2 (*Prior Baseline Benchmarking & Gap Diagnosis*). All of them use the 15 Nov test slates and the scores from Section 3.

### 4.1 How big is the problem? A 2×2 of *cold label* × *article in the fit vocabulary*

The cold label (< 10 prior impressions, from step 1) and "the model has an embedding for this article" are different things. An article first shown on 14 Nov is **warm** by the label on 15 Nov, yet a model fitted on 9–13 Nov has **no embedding** for it. So there are two failure sources: rarely seen articles with poorly trained embeddings, and articles the model has never seen at all.

In [ ]:
c = test[COLD_COL].to_numpy().astype(bool); o = test["oov_item"].to_numpy()
code = c.astype(np.int8) * 2 + o.astype(np.int8)          # integer codes keep memory small on ~14M rows
names = {0: "warm, in fit vocab", 1: "warm, NOT in fit vocab", 2: "cold, in fit vocab", 3: "cold, NOT in fit vocab"}
tab = (pd.DataFrame({"code": code, "label": test["label"].to_numpy()}).groupby("code")
       .agg(displays=("label", "size"), clicks=("label", "sum")).rename(index=names))
tab.index.name = "cold_type"
tab["display_share"] = tab["displays"] / tab["displays"].sum()
tab["click_share"] = tab["clicks"] / tab["clicks"].sum()
tab["CTR"] = tab["clicks"] / tab["displays"]
tab["CTR vs average"] = tab["CTR"] / test["label"].mean()
display(tab.round(4))

*What to look for.* If the "NOT in fit vocab" rows hold a small share of displays but a clearly larger share of clicks (CTR vs average > 1), a model that cannot score them loses engagement. This matches the interim finding that newly published articles are rarer but disproportionately clicked.

### 4.2 Performance split by cold label and by "clicked article unseen in fit"

In [ ]:
frame = base_eval(test)
frame["oov_item"] = test["oov_item"].to_numpy().astype(np.int8)
rename = {"cold": "clicked article NOT in fit vocab", "warm": "clicked article in fit vocab", "mixed": "mixed"}
out = {}
for name in ["NeuMF", "NeuMF + ID masking (p=0.1)", "NeuMF + ID masking (p=0.3)", "Random (reference)"]:
    if name not in test_scores:
        continue
    f = frame.assign(score=test_scores[name])
    out[name] = evaluate_by_segment(f, "score", "oov_item").rename(index=rename)["ndcg@10"]
display(pd.DataFrame(out).round(4))

*What to look for.* (a) Compare "clicked article NOT in fit vocab" with the random reference **in both directions**. All unseen articles share one score, so they sit at an arbitrary place relative to seen articles: far *above* random if that constant happens to out-score most seen articles, far *below* if it does not. Either way the model is not judging the articles, it is applying a constant. §4.3 shows which way it falls, so read the two together. (b) ID masking (🔧 I5) trains that constant into an "average article" vector. If cold results stay far below the warm ones, ID-only models simply lack the information, which is the justification for P2/P4/P5's content features.

### 4.3 Score collapse: what does an unknown ID get scored as?

With a zero unknown vector, every unseen article in the same impression gets **the same score**. The table checks that directly, and shows where unseen articles land in the ranking (percentile 0 = top, 1 = bottom; random ≈ 0.5).

In [ ]:
f = frame.assign(score=test_scores["NeuMF"])
g = f[f["oov_item"] == 1].groupby("impression_id")["score"].agg(["nunique", "size"])
g = g[g["size"] >= 2]
print(f"impressions with ≥2 unseen articles: {len(g):,}; "
      f"of these, all unseen articles share ONE score in {(g['nunique'] == 1).mean():.1%}")

r = add_rank(f, "score")
size = r.groupby("impression_id")["rank"].transform("size")
r["pct"] = (r["rank"] - 1) / (size - 1).clip(lower=1)
display(r.groupby(["oov_item", "label"])["pct"].mean().unstack().rename(
    index={0: "article in fit vocab", 1: "article NOT in fit vocab"}, columns={0: "not clicked", 1: "clicked"}).round(3))
del r

### 4.4 Fair exposure: how many cold articles reach the top 10?

The business objective is to give new and rarely shown articles a fair chance. This compares the cold share of each model's top-10 lists with the cold share of the candidates and of the clicks (only impressions with more than 10 candidates, as in the interim's re-ranking scope).

In [ ]:
cat_map = news.set_index("news_id")["category"] if "category" in news else pd.Series(dtype=object)

def top10_cold_share(frame_scored):
    top = topk(add_rank(frame_scored, "score"), "rank", k=config.TOP_K)
    top["category"] = top["news_id"].map(cat_map).fillna("unknown")
    return topk_diversity(top, "category", None, cold_col=COLD_COL, k=config.TOP_K)

slate = test.groupby("impression_id")["label"].transform("size")
long_ = test[slate > config.TOP_K]
cand_cold = long_.groupby("impression_id")[COLD_COL].mean().mean()
click_cold = long_.loc[long_["label"] == 1, COLD_COL].mean()

expo = {}
for name in [n for n in ["NeuMF", best_masked, "Random (reference)"] if n]:
    expo[name] = top10_cold_share(base_eval(test).assign(score=test_scores[name]))
expo = pd.DataFrame(expo).T[["cold_share", "unique_categories", "n_lists"]]
expo["cold share of candidates"], expo["cold share of clicks"] = cand_cold, click_cold
display(expo.round(4))

*What to look for.* A top-10 cold share well **below** the candidate share means the model systematically buries cold articles. Random sits at the candidate share by construction. P6's MMR sweep then reports the same column after re-ranking.

### 4.5 Popularity dominance: is NeuMF mostly a popularity ranker?

Rank correlation between the model's test score and how many clicks the article collected in fit (articles in the fit vocabulary only).

In [ ]:
from scipy.stats import spearmanr
clicks_per_item = fit.groupby("i")["label"].sum()
pop = np.log1p(test["i"].map(clicks_per_item).fillna(0).to_numpy())
idx = np.flatnonzero(~test["oov_item"].to_numpy())
samp = rng.choice(idx, size=min(500_000, len(idx)), replace=False)
for name in [n for n in ["GMF (MF)", "NCF-MLP", "NeuMF"] if n in test_scores]:
    print(f"Spearman(score, log fit clicks) {name:10s}: {spearmanr(test_scores[name][samp], pop[samp]).correlation:.3f}")

*What to look for.* A high correlation (say > 0.5) means much of the ranking is "show what was clicked before", which is why every learned model has to beat P1's popularity baseline to justify its cost. Compare against the popularity row in `results_step2_popularity.csv`.

### 4.6 Restricted-candidate-set effect: replicating Group 10's evaluation filter on our data (I1)

Group 10's UBCF kept only impressions with a known user, scored only candidates known from training, and dropped impressions left without a click or a non-click. Applying the **same filter** to our test slates (a filter, not a re-sample) shows how much it flatters a model and which clicks it hides.

In [ ]:
keep = (~test["oov_user"] & ~test["oov_item"]).to_numpy()
full_n = test["impression_id"].nunique()
click = test["label"].to_numpy() == 1
cold = test[COLD_COL].to_numpy().astype(bool)
print(f"candidate rows kept by the filter: {keep.mean():.1%}")
print(f"clicks removed by the filter: {(click & ~keep).sum() / click.sum():.1%}; "
      f"cold share of removed clicks {cold[click & ~keep].mean():.1%} vs kept clicks {cold[click & keep].mean():.1%}")

rows_f = []
for name in ["NeuMF", "Random (reference)"]:
    f = base_eval(test).assign(score=test_scores[name])
    m_full = evaluate(f, "score")
    m_filt = evaluate(f[keep], "score")
    rows_f.append({"model": name, "protocol": "full slates (ours)", "impressions evaluated": f"{m_full['n_impressions'] / full_n:.1%}",
                   "auc": m_full["auc"], "mrr": m_full["mrr"], "ndcg@10": m_full["ndcg@10"]})
    rows_f.append({"model": name, "protocol": "Group 10-style filter", "impressions evaluated": f"{m_filt['n_impressions'] / full_n:.1%}",
                   "auc": m_filt["auc"], "mrr": m_filt["mrr"], "ndcg@10": m_filt["ndcg@10"]})
display(pd.DataFrame(rows_f).round(4))

*What to look for.* Under the filter the metrics rise and most impressions disappear, because the impressions that drop out are exactly the ones containing unseen (often cold) articles. Even a random scorer looks different once candidates are removed. Group 10's reported UBCF numbers (Section 5) were produced this way.

## 5. Past group (Group 10): put their reported number next to ours and explain the difference (MIND-large, full slates, cold/warm split).

Puts Group 10's saved results next to ours and shows, with a small experiment, why they cannot be compared directly.


**Why the numbers are not comparable (all checked in Group 10's code and saved outputs):**
1. **Dataset and split.** Group 10 used MIND-*small* with their own temporal split, about 30,000 validation and test impressions. We use MIND-*large*, fit 9–13 Nov, tune 14 Nov, test 15 Nov (hundreds of thousands of impressions).
2. **Filtered evaluation (I1).** Their UBCF skipped 67.7% of validation and 90.4% of test impressions (see §4.6 for the same filter on our data). The remaining impressions have a known user and known articles, so they are the easy ones.
3. **Tie handling (I2).** In their UBCF and CBF notebooks, MRR / nDCG / MAP sort `(score, label)` pairs in descending order, so among equal scores the clicked item comes first. Demonstrated below. (Their MLP sorts on the score alone, so ties keep the input order, which also is not random.)
4. **AUC definition (I3).** Their MLP notebook computes one AUC over all rows flattened together. Ours (and the MIND convention in the interim report) is per impression, then averaged.
5. **MRR definition.** Their MRR is the reciprocal rank of the *first* click. The shared metric averages 1/rank over *all* clicked items, as in the official MIND evaluation. These differ whenever an impression has more than one click (27.7% do).
6. **Cold vs warm.** Group 10 did not split results by cold/warm, so they cannot show the failure this notebook diagnoses.
7. **Scope.** This notebook implements the ID-only GMF / NCF-MLP / NeuMF baselines. The Group 10 rows below are copied from their saved outputs; this notebook does not rerun their UBCF model.

In [ ]:
# Group 10's saved results, copied from the outputs stored in their notebooks
g10 = pd.DataFrame([
    ("UBCF",            "val",  "32.3% (10,213 of 31,624)", 0.5525, 0.4841, 0.5097, 0.5743, "User_Based_CF.ipynb, cell 19"),
    ("UBCF",            "test", "9.6% (2,898 of 30,270)",   0.5680, 0.6338, 0.6836, 0.7159, "User_Based_CF.ipynb, cell 20"),
    ("CBF",             "val",  "100%",                     0.5870, 0.2921, 0.2950, 0.3464, "Content_Based_Filtering_Model.ipynb, cell 23"),
    ("CBF",             "test", "100%",                     0.5782, 0.2825, 0.2836, 0.3373, "Content_Based_Filtering_Model.ipynb, cell 24"),
    ("MLP (final, run A)", "test", "not stated",            0.7103, 0.3077, 0.2803, 0.3418, "5_MLP_model.ipynb, cell 98 (AUC is global, not per impression)"),
    ("MLP (final, run B)", "test", "not stated",            0.7152, 0.2984, 0.2716, 0.3346, "5_MLP_model.ipynb, cell 106 (AUC is global, not per impression)"),
], columns=["model", "split", "impressions evaluated", "auc", "mrr", "ndcg@5", "ndcg@10", "source"])
g10.insert(0, "group", "Group 10 (MIND-small)")

ours = results[(results["split"].str.startswith("test")) & (results["segment"] == "all")].copy()
ours.insert(0, "group", "Group 4 (MIND-large)")
ours["split"] = "test"
ours["impressions evaluated"] = (ours["n_impressions"] / full_n).map("{:.1%}".format)
ours["source"] = "this notebook, full slates, shared metrics"
cols = ["group", "model", "split", "impressions evaluated", "auc", "mrr", "ndcg@5", "ndcg@10", "source"]
display(pd.concat([g10[cols], ours[cols]], ignore_index=True))